# 01 · Mercado de captaciones y colocaciones

Exploración interactiva de los modelos de `docs/analisis_mercado.md`. El pipeline completo está en `scripts/analisis_mercado.py`; acá se usan las mismas funciones para mirar un tema a la vez.

Reglas que aplican: tipo de entidad siempre explícito (regla 1), tasas ponderadas por monto (4), cantón NACIONAL fuera de los mercados geográficos (5), entidades por `entidad_codigo` (3).

In [ ]:
from benchmark_analisis import demanda, liderazgo, mercado, segmentacion, sustitucion, tasas
from benchmark_analisis.paneles import panel

saldos = panel('saldos_canton')
capt = panel('tasas_captacion_semanal')
canton_mes = panel('tasas_canton_mensual')
indic = panel('indicadores_entidad')

## Tasas a mezcla de plazo constante

In [ ]:
w = tasas.series_semanales(capt)
w[liderazgo.GRANDES + ['COOP SEGMENTO 1']].resample('QE').mean().loc['2021':].round(2)

## Sustitución bancos ↔ cooperativas

In [ ]:
M = sustitucion.serie_mensual(w, capt, saldos)
print(sustitucion.modelo_flujos(M)['resumen'])
print(sustitucion.modelo_saldos(M)['resumen'])

## Liderazgo y anticipación

In [ ]:
R, base, N = liderazgo.matriz_respuesta(w.loc['2017':], liderazgo.COMPETIDORES + ['COOP SEGMENTO 1'])
R.round(2)

In [ ]:
D = liderazgo.dataset_movimientos(w)
liderazgo.evaluar_anticipacion(D, 'y_sube')

## Demanda por cantón (depósitos a plazo)

In [ ]:
dep = demanda.preparar(canton_mes, 'DEP_PLAZO')
res = demanda.estimar(dep, 'DEP_PLAZO', anidado=True)
print(res.resumen)

## Cantones y oportunidades de una entidad

In [ ]:
ind = mercado.indicadores_canton(saldos, canton_mes)
cl, sil = mercado.clusters_canton(ind)
mercado.oportunidades_entidad(saldos, cl, 'GUAYAQUIL', 'depositos').head(15)

## Segmentación de entidades

In [ ]:
ent = segmentacion.universo(segmentacion.tabla_entidades(indic, saldos, int(saldos['anio_mes'].max())))
ce, sil_e, Z = segmentacion.agrupar(ent)
segmentacion.pares_cercanos(Z, 'GUAYAQUIL')